In [2]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    RocCurveDisplay, log_loss
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CM_DIR = OUTPUTS_DIR / "confusion_matrices"
ROC_DIR = OUTPUTS_DIR / "roc_curves"
TABLE_DIR = OUTPUTS_DIR / "comparison_tables"
REPORT_DIR = OUTPUTS_DIR / "classification_reports"
for p in [MODELS_DIR, CM_DIR, ROC_DIR, TABLE_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BASELINE_41_DIR = Path.cwd() / "outputs" / "baseline_41features"
CM_DIR = BASELINE_41_DIR / "confusion_matrices"
ROC_DIR = BASELINE_41_DIR / "roc_curves"

for folder in [BASELINE_41_DIR, CM_DIR, ROC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Confusion matrices will be saved to: {CM_DIR}")
print(f"ROC curves will be saved to: {ROC_DIR}")

X_train = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("+", "plus")
        .replace("[", "")
        .replace("]", "")
        .replace(":", "")
    )

def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return model.predict(X)

def optimize_threshold(y_true, scores, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_threshold = 0.5
    best_value = -1
    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        value = f1_score(y_true, preds, zero_division=0) if metric == "f1" else fbeta_score(y_true, preds, beta=2, zero_division=0)
        if value > best_value:
            best_value = value
            best_threshold = threshold
    return float(best_threshold), float(best_value)

def metric_block(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    out = {
        "Accuracy": accuracy_score(y_true, preds),
        "Error Rate": 1 - accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "F2": fbeta_score(y_true, preds, beta=2, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores),
    }
    try:
        clipped = np.clip(scores, 1e-6, 1 - 1e-6)
        out["Log Loss"] = log_loss(y_true, clipped)
    except Exception:
        out["Log Loss"] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    out.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return out, preds

def plot_confusion(y_true, preds, title, path):
    cm = confusion_matrix(y_true, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_roc(y_true, scores, title, path):
    plt.figure(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_true, scores)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def evaluate_model(model, model_name, phase, threshold_source="train"):
    model.fit(X_train, y_train)
    train_scores = predict_scores(model, X_train)
    test_scores = predict_scores(model, X_test)
    threshold, threshold_metric = optimize_threshold(y_train, train_scores, metric="f1")
    train_metrics, train_preds = metric_block(y_train, train_scores, threshold)
    test_metrics, test_preds = metric_block(y_test, test_scores, threshold)
    cv_acc = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1)
    cv_f1 = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="f1", n_jobs=1)
    try:
        cv_auc = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    except Exception:
        cv_auc = np.array([np.nan])

    name = safe_name(f"{phase}_{model_name}")
    plot_confusion(y_test, test_preds, f"{phase} {model_name} Confusion Matrix", CM_DIR / f"{name}_cm.png")
    plot_roc(y_test, test_scores, f"{phase} {model_name} ROC Curve", ROC_DIR / f"{name}_roc.png")
    (REPORT_DIR / f"{name}_classification_report.txt").write_text(classification_report(y_test, test_preds, target_names=["No PCOS", "PCOS"], zero_division=0))
    joblib.dump(model, MODELS_DIR / f"{name}.joblib")
    (MODELS_DIR / f"{name}_features.json").write_text(    # ADD THIS
    json.dumps(list(X_train.columns))
)

    row = {
        "Model": model_name,
        "Phase": phase,
        "Threshold": threshold,
        "CV Accuracy Mean": np.nanmean(cv_acc),
        "CV Accuracy Std": np.nanstd(cv_acc),
        "CV F1 Mean": np.nanmean(cv_f1),
        "CV F1 Std": np.nanstd(cv_f1),
        "CV ROC-AUC Mean": np.nanmean(cv_auc),
        "CV ROC-AUC Std": np.nanstd(cv_auc),
    }
    row.update({f"Train {k}": v for k, v in train_metrics.items()})
    row.update({f"Test {k}": v for k, v in test_metrics.items()})
    return row, model, test_scores

Confusion matrices will be saved to: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\notebooks\outputs\baseline_41features\confusion_matrices
ROC curves will be saved to: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\notebooks\outputs\baseline_41features\roc_curves


In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
    "Gaussian NB": GaussianNB(),
    "Bagging": BaggingClassifier(estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42), n_estimators=150, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=150, learning_rate=0.5, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=21),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),
    "Perceptron": CalibratedClassifierCV(Perceptron(class_weight="balanced", random_state=42), cv=5),
}
if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42
    )
else:
    models["XGBoost"] = GradientBoostingClassifier(random_state=43)

models["Stacking ML"] = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    cv=5,
    n_jobs=1,
)

print(f"Total baseline models: {len(models)}")
list(models)

Total baseline models: 14


['Logistic Regression',
 'Decision Tree',
 'Random Forest',
 'SVM',
 'Gaussian NB',
 'Bagging',
 'AdaBoost',
 'Gradient Boosting',
 'KNN',
 'LDA',
 'QDA',
 'Perceptron',
 'XGBoost',
 'Stacking ML']

In [5]:
results = []
fitted_models = {}
roc_scores = {}

for name, model in models.items():
    print(f"Training baseline model: {name}")
    row, fitted, test_scores = evaluate_model(model, name, "Baseline")
    results.append(row)
    fitted_models[name] = fitted
    roc_scores[name] = test_scores

baseline_results = pd.DataFrame(results)
metric_cols = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
baseline_results["Rank"] = baseline_results["Test F1"].rank(ascending=False, method="min").astype(int)
baseline_results = baseline_results.sort_values(["Rank", "Test ROC-AUC"], ascending=[True, False])
baseline_results.to_csv(TABLE_DIR / "baseline_results.csv", index=False)
baseline_results.to_csv(TABLE_DIR / "baseline_model_results.csv", index=False)
display(baseline_results[["Rank", "Model", "Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC", "CV Accuracy Mean"]])

Training baseline model: Logistic Regression
Training baseline model: Decision Tree
Training baseline model: Random Forest
Training baseline model: SVM
Training baseline model: Gaussian NB
Training baseline model: Bagging
Training baseline model: AdaBoost
Training baseline model: Gradient Boosting
Training baseline model: KNN
Training baseline model: LDA
Training baseline model: QDA
Training baseline model: Perceptron
Training baseline model: XGBoost
Training baseline model: Stacking ML


,Rank,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,CV Accuracy Mean
6,1,AdaBoost,0.895706,0.860000,0.811321,0.834951,0.948714,0.875462
9,2,LDA,0.883436,0.793103,0.867925,0.828829,0.949228,0.891465
12,3,XGBoost,0.877301,0.789474,0.849057,0.818182,0.946998,0.894026
5,3,Bagging,0.877301,0.789474,0.849057,0.818182,0.932419,0.878378
13,5,Stacking ML,0.883436,0.840000,0.792453,0.815534,0.949400,0.886131
2,6,Random Forest,0.865031,0.738462,0.905660,0.813559,0.948285,0.886273
1,7,Decision Tree,0.865031,0.781818,0.811321,0.796296,0.851115,0.785491
7,8,Gradient Boosting,0.852761,0.745763,0.830189,0.785714,0.945283,0.896871
0,9,Logistic Regression,0.865031,0.816327,0.754717,0.784314,0.936535,0.878236
11,10,Perceptron,0.858896,0.788462,0.773585,0.780952,0.919039,0.870270


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

...

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, log_loss, RocCurveDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"

BASE_OUT  = PROJECT_ROOT / "outputs" / "without_FE_baseline"
TABLE_DIR = PROJECT_ROOT / "outputs" / "comparison_tables"

for p in [MODELS_DIR, TABLE_DIR, BASE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Output root:", BASE_OUT)

Output root: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline


In [2]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test  = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Target distribution (train):\n{y_train.value_counts()}")

Train : (378, 41)
Test  : (163, 41)
Target distribution (train):
PCOS
0    254
1    124
Name: count, dtype: int64


In [ ]:
def safe_name(name):
    return (name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "")
            .replace("+", "plus").replace(":", ""))


def model_dir(model_name):
    d = BASE_OUT / safe_name(model_name)
    d.mkdir(parents=True, exist_ok=True)
    return d


def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X).astype(float)


def optimize_threshold(y_true, scores, metric="f1"):
    best_t, best_v = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (scores >= t).astype(int)
        v = (f1_score(y_true, preds, zero_division=0) if metric == "f1"
             else fbeta_score(y_true, preds, beta=2, zero_division=0))
        if v > best_v:
            best_v, best_t = v, t
    return float(best_t), float(best_v)


def metric_block(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    out = {
        "Accuracy"  : accuracy_score(y_true, preds),
        "Error Rate": 1 - accuracy_score(y_true, preds),
        "Precision" : precision_score(y_true, preds, zero_division=0),
        "Recall"    : recall_score(y_true, preds, zero_division=0),
        "F1"        : f1_score(y_true, preds, zero_division=0),
        "F2"        : fbeta_score(y_true, preds, beta=2, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_true, scores),
        "PR-AUC"    : average_precision_score(y_true, scores),
    }
    try:
        out["Log Loss"] = log_loss(y_true, np.clip(scores, 1e-6, 1 - 1e-6))
    except Exception:
        out["Log Loss"] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    out.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return out, preds


def plot_confusion(y_true, preds, model_name, out_dir):
    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax)
    ax.set_title(f"without FE Baseline — {model_name}\nConfusion Matrix", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    path = out_dir / "confusion_matrix.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


# ── Plot: ROC Curve ────────────────────────────────────────────────────────
def plot_roc(y_true, scores, model_name, out_dir):
    auc = roc_auc_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # 1. Pass an empty or generic string to name to prevent standard concatenation conflicts
    disp = RocCurveDisplay.from_predictions(y_true, scores, name="", ax=ax)
    
    # 2. Directly overwrite the line label with your precise 4-decimal string
    disp.line_.set_label(f"Classifier (AUC = {auc:.4f})")
    
    # 3. Re-draw the legend to apply your custom label formatting
    ax.legend(loc="lower right")
    
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(f"without FE Baseline — {model_name}\nROC Curve", fontsize=11, fontweight="bold")
    plt.tight_layout()
    path = out_dir / "roc_curve.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def plot_stratified_cv(model, model_name, out_dir):
    fold_acc, fold_f1, fold_auc = [], [], []
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    for fold_i, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
        m = clone(model)
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        sc  = predict_scores(m, X_train.iloc[va])
        th, _ = optimize_threshold(y_train.iloc[tr],
                                   predict_scores(m, X_train.iloc[tr]))
        preds = (sc >= th).astype(int)
        fold_acc.append(accuracy_score(y_train.iloc[va], preds))
        fold_f1.append(f1_score(y_train.iloc[va], preds, zero_division=0))
        try:
            fold_auc.append(roc_auc_score(y_train.iloc[va], sc))
        except Exception:
            fold_auc.append(np.nan)

    folds = np.arange(1, 11)
    fig, ax = plt.subplots(figsize=(12, 5))
    for vals, label, marker in [
        (fold_acc, f"Accuracy (mean={np.nanmean(fold_acc):.4f})", "o"),
        (fold_f1,  f"F1       (mean={np.nanmean(fold_f1):.4f})",  "s"),
        (fold_auc, f"ROC-AUC  (mean={np.nanmean(fold_auc):.4f})", "^"),
    ]:
        ax.plot(folds, vals, marker=marker, linewidth=1.8, label=label)
        for x, y in zip(folds, vals):
            ax.annotate(f"{y:.4f}", (x, y),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=7.5, color="dimgray")

    ax.set_xticks(folds)
    ax.set_xticklabels([f"Fold {i}" for i in folds])
    ax.set_ylabel("Score")
    ax.set_ylim(max(0, min(fold_acc + fold_f1 + fold_auc) - 0.08), 1.08)
    ax.set_title(
        f"without FE Baseline — {model_name}\n"
        f"10-Fold Stratified CV ({X_train.shape[1]} Features)",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    path = out_dir / "stratified_cv.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

print("Helper functions defined.")

Helper functions defined.


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42),

    "SVM": SVC(
        kernel="rbf", probability=True, class_weight="balanced", random_state=42),

    "Gaussian NB": GaussianNB(),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
        n_estimators=150, random_state=42),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=150, learning_rate=0.5, random_state=42),

    "Gradient Boosting": GradientBoostingClassifier(random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=21),

    "LDA": LinearDiscriminantAnalysis(),

    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),

    "Perceptron": CalibratedClassifierCV(
        Perceptron(class_weight="balanced", random_state=42), cv=5),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42)
else:
    models["XGBoost"] = GradientBoostingClassifier(random_state=43)

models["Stacking ML"] = StackingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ("gb",  GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42),
    cv=5, n_jobs=1,
)

print(f"Total models: {len(models)}")
print(list(models.keys()))

Total models: 14
['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'Gaussian NB', 'Bagging', 'AdaBoost', 'Gradient Boosting', 'KNN', 'LDA', 'QDA', 'Perceptron', 'XGBoost', 'Stacking ML']


In [5]:
results = []

for name, model in models.items():
    print(f"\n▶ Training : {name}")
    t_train_start = time.time()
    model.fit(X_train, y_train)
    train_time_sec = round(time.time() - t_train_start, 4)
    out_dir = model_dir(name)

    model.fit(X_train, y_train)
    train_scores = predict_scores(model, X_train)
    test_scores  = predict_scores(model, X_test)

    threshold, _ = optimize_threshold(y_train, train_scores, metric="f1")

    train_metrics, train_preds = metric_block(y_train, train_scores, threshold)
    test_metrics,  test_preds  = metric_block(y_test,  test_scores,  threshold)

    cv_acc = cross_val_score(clone(model), X_train, y_train,
                             cv=cv, scoring="accuracy", n_jobs=1)
    cv_f1  = cross_val_score(clone(model), X_train, y_train,
                             cv=cv, scoring="f1",       n_jobs=1)
    try:
        cv_auc = cross_val_score(clone(model), X_train, y_train,
                                 cv=cv, scoring="roc_auc", n_jobs=1)
    except Exception:
        cv_auc = np.array([np.nan])

    plot_confusion(y_test, test_preds,  name, out_dir)
    plot_roc(y_test,       test_scores, name, out_dir)
    plot_stratified_cv(model,           name, out_dir)

    joblib.dump(model, MODELS_DIR / f"without_fe_baseline_{safe_name(name)}.joblib")
    (MODELS_DIR / f"without_fe_baseline_{safe_name(name)}_features.json").write_text(
        json.dumps(list(X_train.columns)))

    row = {
        "Model"           : name,
        "Threshold"       : threshold,
        "CV Accuracy Mean": np.nanmean(cv_acc),
        "CV Accuracy Std" : np.nanstd(cv_acc),
        "CV F1 Mean"      : np.nanmean(cv_f1),
        "CV F1 Std"       : np.nanstd(cv_f1),
        "CV ROC-AUC Mean" : np.nanmean(cv_auc),
        "CV ROC-AUC Std"  : np.nanstd(cv_auc),
        "train_time_sec"  : train_time_sec,
    }
    row.update({f"Train {k}": v for k, v in train_metrics.items()})
    row.update({f"Test {k}" : v for k, v in test_metrics.items()})
    results.append(row)
    print(f"  ✓ Done | Acc={test_metrics['Accuracy']:.4f} "
          f"F1={test_metrics['F1']:.4f} AUC={test_metrics['ROC-AUC']:.4f}")


▶ Training : Logistic Regression
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline\logistic_regression\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline\logistic_regression\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline\logistic_regression\stratified_cv.png
  ✓ Done | Acc=0.8650 F1=0.7843 AUC=0.9365

▶ Training : Decision Tree
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline\decision_tree\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_baseline\decision_tree\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\without_FE_ba

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Define explicit base directory dynamically
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
BASE_OUT.mkdir(parents=True, exist_ok=True)

# 2. Build the results DataFrame directly from your active notebook run
df_results = pd.DataFrame(results)
df_results["Rank"] = df_results["Test F1"].rank(ascending=False, method="min").astype(int)
df_results = df_results.sort_values(["Rank", "Test ROC-AUC"], ascending=[True, False])

# 3. Display the tuned performance table
display(df_results[[
    "Rank", "Model",
    "Test Accuracy", "Test Precision", "Test Recall",
    "Test F1", "Test ROC-AUC",
    "CV F1 Mean", "CV ROC-AUC Mean"
]])

# 4. Save your brand new tuned outputs cleanly to disk
df_results.to_csv(BASE_OUT / "without_FE_Baseline_results.csv", index=False)

print("\nAll done. Outputs saved under:", BASE_OUT)


,Rank,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,CV F1 Mean,CV ROC-AUC Mean
6,1,AdaBoost,0.895706,0.860000,0.811321,0.834951,0.948714,0.800833,0.932551
9,2,LDA,0.883436,0.793103,0.867925,0.828829,0.949228,0.824073,0.934821
12,3,XGBoost,0.877301,0.789474,0.849057,0.818182,0.946998,0.831067,0.945436
5,3,Bagging,0.877301,0.789474,0.849057,0.818182,0.932419,0.802949,0.941378
13,5,Stacking ML,0.883436,0.840000,0.792453,0.815534,0.949400,0.830392,0.953962
2,6,Random Forest,0.865031,0.738462,0.905660,0.813559,0.948285,0.806946,0.954397
1,7,Decision Tree,0.865031,0.781818,0.811321,0.796296,0.851115,0.666353,0.754397
7,8,Gradient Boosting,0.852761,0.745763,0.830189,0.785714,0.945283,0.836336,0.950282
0,9,Logistic Regression,0.865031,0.816327,0.754717,0.784314,0.936535,0.818170,0.933731
11,10,Perceptron,0.858896,0.788462,0.773585,0.780952,0.919039,0.770987,0.928859



All done. Outputs saved under: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs
